# Notebook 15b — Correctness Tests 3 and 4 at the Selected Configuration

**Hyperparameter-optimisation experiment · Soccer Event Transformer · Track A model family**

| Item | Content |
|---|---|
| **Question** | Do correctness tests 3 and 4 pass at the selected configuration once the sequence length is taken from the model instead of a fixed value? |
| **The two tests** | **Test 3** checks that the hand-written causal multi-head attention equals PyTorch's reference `scaled_dot_product_attention` (with `is_causal=True`) on random inputs. **Test 4** checks, on real validation windows, that every row of attention probabilities sums to 1 and that every weight on a later position is exactly 0. |
| **Why this notebook exists** | When the 16 correctness tests in `tests/test_model.py` were executed at the selected configuration (4 layers, 2 heads, d_model 128, d_ff 512, K = 16, T300), tests 3 and 4 raised shape errors. Their code fixes the sequence length at 32, the development window; the selected model's window is 16. The errors come from that literal, not from the model. The other 14 tests passed. |
| **What is done** | Tests 3 and 4 are re-executed with one change only: the literal 32 is replaced by `T = model.config.context_length`. Every other line is copied from `tests/test_model.py`, and the difference is printed from the source. The original module is not modified; the original tests are also re-run, to record their error. |
| **Data** | No training and no test data. Random inputs (test 3) and validation windows (test 4), as in the original tests. |
| **Outputs** | `final_benchmark/results/15b_correctness_tests_3_4.json` and `final_benchmark/results/correctness_tests_final_configuration_complete.csv` (the full 16-test table with tests 3 and 4 updated). |

**Check that the tested model is the selected model.** The test model is built from the frozen configuration; its parameter count (911,119) must equal the count recorded for the five final runs.

In [ ]:
# ---------------------------------------------------------------------------
# Environment setup (same layout as notebooks 08-14 of this experiment)
# ---------------------------------------------------------------------------
import os
import sys
import json
import time
import shutil
import hashlib
import importlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_DIR = Path(os.environ.get("TRACK_A_DIR", "/content/drive/MyDrive/soccer_mi/track_a"))
RUN_MODE = os.environ.get("TRACK_A_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

EXISTING_DIR = PROJECT_DIR if RUN_MODE == "full" else PROJECT_DIR / "smoke"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
EXISTING_RUNS_DIR = EXISTING_DIR / "runs"
EXISTING_RESULTS_DIR = EXISTING_DIR / "results"
EXISTING_FROZEN_CONFIG = EXISTING_DIR / "config" / "frozen_config.json"
BENCH_DIR = EXISTING_DIR / "benchmarks" / "track_a_4layer"
OPT_DIR = EXISTING_DIR / "experiments" / "hyperparameter_optimisation"
OPT_CONFIG_PATH = OPT_DIR / "config" / "frozen_config.json"

WORK_DIR = OPT_DIR / "final_benchmark"
RUNS_DIR = WORK_DIR / "runs"
RESULTS_DIR = WORK_DIR / "results"
FIGURES_DIR = WORK_DIR / "figures"
CONFIG_DIR = WORK_DIR / "config"
MANIFEST_DIR = WORK_DIR / "manifests"
LOG_DIR = WORK_DIR / "logs"
for folder in [RUNS_DIR, RESULTS_DIR, FIGURES_DIR, CONFIG_DIR, MANIFEST_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = MANIFEST_DIR / "experiment_manifest.jsonl"


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path) as handle:
        return json.load(handle)


print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("Outputs of this notebook:", WORK_DIR)

Mounted at /content/drive
Colab: True | RUN_MODE: full
Outputs of this notebook: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.batching as batching
import src.metrics as metrics
import src.model as model_module
import src.training as training
import src.analysis as analysis
import src.evaluation as evaluation
for module in [data, batching, metrics, model_module, training, analysis, evaluation]:
    importlib.reload(module)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_BOOT = 2000 if RUN_MODE == "full" else 50

events = pd.read_parquet(ARTIFACTS_DIR / "events_prepared.parquet")
targets = events["action_id"].to_numpy().astype("int64")
match_ids = events["match_id"].to_numpy()

TRACK_A_FROZEN = load_json(EXISTING_FROZEN_CONFIG)
TRACK_A_HYPERPARAMETERS = load_json(EXISTING_RESULTS_DIR / "04_selected_hyperparameters.json")
SELECTED = load_json(OPT_CONFIG_PATH)

ENCODING = TRACK_A_FROZEN["encoding"]
POSSESSION_FEATURES = TRACK_A_FROZEN["possession_features"]
STRIDE = TRACK_A_FROZEN["stride"]
FINAL_CONTEXT = SELECTED["context_length"]
FINAL_SEEDS = SELECTED["seed_policy"]["final_seeds"]

print("Events:", len(events), "| device:", DEVICE)
print("Selected configuration (notebook 12):",
      json.dumps({key: SELECTED[key] for key in ["context_length", "model", "training"]}))
print("Final seeds:", FINAL_SEEDS)

Events: 658905 | device: cpu
Selected configuration (notebook 12): {"context_length": 16, "model": {"n_layers": 4, "n_heads": 2, "d_model": 128, "d_ff": 512}, "training": {"learning_rate": 0.0003, "weight_decay": 0.01, "max_epochs": 30, "batch_size": 256, "warmup_fraction": 0.05, "min_lr_ratio": 0.1, "grad_clip": 1.0, "patience": 5}}
Final seeds: [100, 101, 102, 103, 104]


In [ ]:
import torch.nn.functional as F
(PROJECT_DIR / "tests" / "__init__.py").touch()
for name in list(sys.modules):
    if name == "tests" or name.startswith("tests."):
        del sys.modules[name]
importlib.invalidate_caches()
import tests.test_model as model_tests
importlib.reload(model_tests)

final_table = pd.read_csv(OPT_RESULTS_DIR / "13_final_runs.csv")
check_encoding = data.load_encoding(ARTIFACTS_DIR / "encodings" / ENCODING)
check_features, check_vocabulary_sizes = data.encode_features(events, check_encoding, possession_features=POSSESSION_FEATURES)
check_encoded = batching.EncodedData(check_features, targets, DEVICE)
check_val_windows = data.build_windows(events, FINAL_CONTEXT, STRIDE, split="val")
SELECTED_MODEL_SETTINGS = {"n_layers": SELECTED["model"]["n_layers"], "n_heads": SELECTED["model"]["n_heads"],
                           "d_model": SELECTED["model"]["d_model"], "d_ff": SELECTED["model"]["d_ff"],
                           "context_length": FINAL_CONTEXT}


def make_batch(n_windows=8, first_window=0):
    rows = torch.as_tensor(check_val_windows[first_window:first_window + n_windows], device=DEVICE)
    return check_encoded.batch(rows)


def make_model(seed=0, **overrides):
    torch.manual_seed(seed)
    settings = dict(SELECTED_MODEL_SETTINGS)
    settings.update(overrides)
    config = model_module.ModelConfig(vocabulary_sizes=check_vocabulary_sizes, **settings)
    return model_module.EventTransformer(config).to(DEVICE)


tested_parameters = model_module.count_parameters(make_model(0))["total"]
recorded_parameters = int(final_table["parameters"].iloc[0])
print(f"Parameters of the test model: {tested_parameters:,} | recorded for the notebook 13 runs: {recorded_parameters:,}")
if tested_parameters != recorded_parameters:
    raise RuntimeError("BLOCKED: the test model is not the selected configuration.")
context = {"make_model": make_model, "make_batch": make_batch}

Parameters of the test model: 911,119 | recorded for the notebook 13 runs: 911,119


## Tests 3 and 4 with the length read from the model

The two functions below are the originals from `tests/test_model.py` with `32` replaced by `T = model.config.context_length` (and `T` added to the printed detail). The unified diff printed under the cell shows that nothing else differs.

**RESULT (next cell).**

| Test | Original code at K = 16 | Length read from the model |
|---|---|---|
| 3 attention vs reference | error: tensor size 16 does not match 32 | PASS: max \|ours − reference\| = 1.12e-08 (T = 16) |
| 4 attention probabilities | error: mask shape [32, 32] does not match the attention tensor | PASS: max \|row sum − 1\| = 2.38e-07; max future weight = 0.0 (T = 16) |

With these two results, all 16 tests pass at the selected configuration, with tests 3 and 4 marked "PASS (length read from model)" in the complete table. The run recorded the device (CPU), the PyTorch version and the SHA-256 of the unchanged test module.


In [ ]:
def test_attention_matches_reference_length_from_model(context):
    model = context["make_model"](0)
    T = model.config.context_length
    attention = model.blocks[0].attention
    torch.manual_seed(1)
    x = torch.randn(4, T, model.config.d_model, device=next(model.parameters()).device)
    with torch.no_grad():
        ours = attention(x)
        queries = attention.split_heads(attention.W_Q(x))
        keys = attention.split_heads(attention.W_K(x))
        values = attention.split_heads(attention.W_V(x))
        reference_heads = F.scaled_dot_product_attention(queries, keys, values, is_causal=True)
        reference = attention.W_O(reference_heads.transpose(1, 2).reshape(4, T, model.config.d_model))
    max_difference = (ours - reference).abs().max().item()
    return max_difference < 1e-5, f"max |ours - reference| = {max_difference:.2e} (T = {T})"


def test_attention_probabilities_length_from_model(context):
    model = context["make_model"](0)
    T = model.config.context_length
    inputs, _ = context["make_batch"](8, 0)
    with torch.no_grad():
        _, internals = model(inputs, return_internals=True)
    worst_row_sum = 0.0
    worst_future = 0.0
    for block in internals["blocks"]:
        probabilities = block["attention_probabilities"]
        worst_row_sum = max(worst_row_sum, (probabilities.sum(-1) - 1).abs().max().item())
        future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=probabilities.device), diagonal=1)
        worst_future = max(worst_future, probabilities[..., future].abs().max().item())
    passed = worst_row_sum < 1e-5 and worst_future == 0.0
    return passed, f"max |row sum - 1| = {worst_row_sum:.2e}; max future weight = {worst_future} (T = {T})"


import inspect
import difflib
for original, adapted in [(model_tests.test_attention_matches_reference, test_attention_matches_reference_length_from_model),
                          (model_tests.test_attention_probabilities, test_attention_probabilities_length_from_model)]:
    diff = difflib.unified_diff(inspect.getsource(original).splitlines(), inspect.getsource(adapted).splitlines(),
                                "tests/test_model.py", "this notebook", lineterm="", n=0)
    print("\n".join(diff))
    print()

--- tests/test_model.py
+++ this notebook
@@ -1 +1 @@
-def test_attention_matches_reference(context):
+def test_attention_matches_reference_length_from_model(context):
@@ -2,0 +3 @@
+    T = model.config.context_length
@@ -5 +6 @@
-    x = torch.randn(4, 32, model.config.d_model, device=next(model.parameters()).device)
+    x = torch.randn(4, T, model.config.d_model, device=next(model.parameters()).device)
@@ -12 +13 @@
-        reference = attention.W_O(reference_heads.transpose(1, 2).reshape(4, 32, model.config.d_model))
+        reference = attention.W_O(reference_heads.transpose(1, 2).reshape(4, T, model.config.d_model))
@@ -14 +15 @@
-    return max_difference < 1e-5, f"max |ours - reference| = {max_difference:.2e}"
+    return max_difference < 1e-5, f"max |ours - reference| = {max_difference:.2e} (T = {T})"

--- tests/test_model.py
+++ this notebook
@@ -1 +1 @@
-def test_attention_probabilities(context):
+def test_attention_probabilities_length_from_model(context):
@@ -2,0 +3 @@


In [ ]:
results = []
for label, original, adapted in [("3 attention vs reference", model_tests.test_attention_matches_reference,
                                  test_attention_matches_reference_length_from_model),
                                 ("4 attention probabilities", model_tests.test_attention_probabilities,
                                  test_attention_probabilities_length_from_model)]:
    try:
        passed_original, detail_original = original(context)
    except Exception as error:
        passed_original, detail_original = False, f"ERROR: {error!r}"
    try:
        passed_adapted, detail_adapted = adapted(context)
    except Exception as error:
        passed_adapted, detail_adapted = False, f"ERROR: {error!r}"
    results.append({"test": label, "original code at final config": "PASS" if passed_original else "FAIL",
                    "original detail": detail_original,
                    "length read from model": "PASS" if passed_adapted else "FAIL", "detail": detail_adapted})
addendum = pd.DataFrame(results).set_index("test")
display(addendum)

complete = pd.read_csv(RESULTS_DIR / "correctness_tests_final_configuration.csv").set_index("test")
for label, row in addendum.iterrows():
    complete.loc[label, "final config (4L_2H_K16, T300)"] = (
        "PASS (length read from model)" if row["length read from model"] == "PASS" else "FAIL")
    complete.loc[label, "detail (final config)"] = row["detail"]
    complete.loc[label, "status"] = "PASS" if row["length read from model"] == "PASS" else "CHECK"
complete.to_csv(RESULTS_DIR / "correctness_tests_final_configuration_complete.csv")
all_pass = bool((complete["status"] == "PASS").all())
training.save_json(RESULTS_DIR / "15b_correctness_tests_3_4.json",
                   {"results": results, "all_16_pass_at_final_config": all_pass,
                    "test_module_sha256": file_sha256(PROJECT_DIR / "tests" / "test_model.py"),
                    "tested_parameters": tested_parameters, "device": DEVICE, "torch": torch.__version__,
                    "written_at": pd.Timestamp.now(tz="UTC").isoformat()})
print(f"All 16 tests pass at the final configuration (tests 3 and 4 with the length read from the model): {all_pass}")
complete[["final config (4L_2H_K16, T300)", "status"]]

,original code at final config,original detail,length read from model,detail
test,,,,
3 attention vs reference,FAIL,ERROR: RuntimeError('The size of tensor a (16) must match the size of tensor b (32) at non-singleton dimension 3'),PASS,max |ours - reference| = 1.12e-08 (T = 16)
4 attention probabilities,FAIL,"ERROR: IndexError('The shape of the mask [32, 32] at index 0 does not match the shape of the indexed tensor [8, 2, 1...",PASS,max |row sum - 1| = 2.38e-07; max future weight = 0.0 (T = 16)


All 16 tests pass at the final configuration (tests 3 and 4 with the length read from the model): True


,"final config (4L_2H_K16, T300)",status
test,,
1 shapes,PASS,PASS
2 causal mask,PASS,PASS
3 attention vs reference,PASS (length read from model),PASS
4 attention probabilities,PASS (length read from model),PASS
5 residual additions,PASS,PASS
6 LayerNorm placement,PASS,PASS
7 positional embeddings,PASS,PASS
8 output dimension,PASS,PASS
9 determinism,PASS,PASS
